# Evaluacion Transversal — Predicción de abandono de clientes en ventas y marketing


**Integrantes:**

- Sebastian Lagos
- Oscar Oreste

**Contexto**

Una empresa del área de ventas y marketing dispone de información demográfica,
comercial y de experiencia de sus clientes. El objetivo del proyecto es analizar
estos datos y desarrollar un modelo de clasificación que permita identificar
clientes con riesgo de abandono.

La unidad de análisis corresponde a cada cliente y la variable objetivo es
`churn`, donde:

- `0`: el cliente no abandonó.
- `1`: el cliente abandonó.

El resultado puede apoyar decisiones de retención, marketing y experiencia del
cliente.

## Objetivos del proyecto

### Objetivo general

Desarrollar una solución reproducible de ciencia de datos para identificar
clientes con riesgo de abandono mediante técnicas de limpieza, integración de
datos, análisis exploratorio, clasificación y visualización interactiva.

### Objetivos específicos

1. Auditar la estructura y calidad del dataset.
2. Aplicar reglas de limpieza reproducibles sin modificar el archivo original.
3. Integrar indicadores externos mediante una API.
4. Analizar las características asociadas al abandono.
5. Comparar modelos de clasificación mediante accuracy.
6. Interpretar los resultados utilizando una matriz de confusión.
7. Construir un dashboard interactivo con Plotly 

## 1. Importación de librerías

Se importan las librerías principales que se utilizarán para cargar, revisar y
trabajar con el dataset.

Las librerías de visualización, API y Machine Learning se incorporarán en las
secciones correspondientes.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Carga del dataset

El dataset se encuentra almacenado en la carpeta `data/raw`.

Se utilizan dos rutas posibles para permitir que el notebook pueda ejecutarse
desde la raíz del repositorio o desde la carpeta `notebook`.

In [4]:
rutas_posibles = [
    Path("data/raw/sales_marketing_customer_dataset.csv"),
    Path("../data/raw/sales_marketing_customer_dataset.csv")
]

ruta_dataset = next(
    (ruta for ruta in rutas_posibles if ruta.exists()),
    None
)

if ruta_dataset is None:
    raise FileNotFoundError(
        "No se encontró sales_marketing_customer_dataset.csv en data/raw."
    )

df = pd.read_csv(ruta_dataset)

print("Dataset cargado correctamente.")
print("Ruta utilizada:", ruta_dataset)

Dataset cargado correctamente.
Ruta utilizada: ..\data\raw\sales_marketing_customer_dataset.csv


## 3. Vista inicial del dataset

Se muestran las primeras filas para conocer la estructura general y observar
ejemplos de los datos disponibles.

In [5]:
df.head()

,customer_id,gender,age,country,city,signup_date,last_purchase_date,acquisition_channel,device_type,subscription_type,is_premium_user,total_visits,avg_session_time,pages_per_session,email_open_rate,email_click_rate,total_spent,avg_order_value,discount_used,coupon_code,support_tickets,refund_requested,delivery_delay_days,payment_method,satisfaction_score,nps_score,marketing_spend_per_user,lifetime_value,last_3_month_purchase_freq,churn
0,10001,Male,52.00,India,Berlin,2022-05-10 00:00:00,2024-12-31 00:00:00,Email,Tablet,Annual,1,7,13.90,5.42,0.67,0.26,559.52,65.25,0,NEW20,0,0,3,UPI,3.00,10,27.56,915.31,14,0
1,10002,NaN,35.00,Germany,Mumbai,2024-06-16 00:00:00,2024-05-07 00:00:00,Organic,Desktop,Monthly,0,19,5.11,5.35,0.70,0.37,356.49,48.47,1,NEW20,5,0,3,BKash,3.00,7,15.15,"2,079.96",11,0
2,10003,Female,27.00,Germany,London,2023-08-23 00:00:00,2024-04-28 00:00:00,Email,Mobile,Annual,1,18,9.74,3.59,0.47,0.44,689.33,77.82,0,NaN,1,0,2,UPI,5.00,6,13.51,"1,379.15",9,0
3,10004,Female,36.00,India,Mumbai,2024-01-28 00:00:00,2023-05-20 00:00:00,Facebook Ads,Tablet,Annual,1,16,9.64,2.95,0.58,0.37,445.43,71.71,0,NaN,0,0,2,PayPal,4.00,6,25.65,774.65,7,0
4,10005,Male,29.00,USA,Hamburg,2023-07-21 00:00:00,2024-04-07 00:00:00,Referral,Mobile,Monthly,0,12,7.79,2.41,0.05,0.16,686.29,44.99,1,NaN,2,1,4,BKash,3.00,1,12.39,87.68,11,0


## 4. Inspección inicial del dataset

Se revisan las dimensiones, los nombres de las columnas y los tipos de datos.

Esta inspección permite comprender la estructura antes de aplicar cualquier
limpieza o transformación.

In [6]:
print("Dimensión del dataset (filas, columnas):")
print(df.shape)

print("\nColumnas del dataset:")
print(df.columns.tolist())

print("\nInformación general:")
df.info()

Dimensión del dataset (filas, columnas):
(15000, 30)

Columnas del dataset:
['customer_id', 'gender', 'age', 'country', 'city', 'signup_date', 'last_purchase_date', 'acquisition_channel', 'device_type', 'subscription_type', 'is_premium_user', 'total_visits', 'avg_session_time', 'pages_per_session', 'email_open_rate', 'email_click_rate', 'total_spent', 'avg_order_value', 'discount_used', 'coupon_code', 'support_tickets', 'refund_requested', 'delivery_delay_days', 'payment_method', 'satisfaction_score', 'nps_score', 'marketing_spend_per_user', 'lifetime_value', 'last_3_month_purchase_freq', 'churn']

Información general:
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customer_id                 15000 non-null  int64  
 1   gender                      14262 non-null  str    
 2   age                         13800 non-null

## 5. Resumen estadístico inicial

Se calculan estadísticas descriptivas para las variables numéricas.

Esto permite observar sus valores mínimos, máximos, promedios y posibles
diferencias de escala.

In [7]:
print("Resumen estadístico de las columnas numéricas:")
display(df.describe().T)

Resumen estadístico de las columnas numéricas:


,count,mean,std,min,25%,50%,75%,max
customer_id,"15,000.00","17,500.50","4,330.27","10,001.00","13,750.75","17,500.50","21,250.25","25,000.00"
age,"13,800.00",35.20,10.33,-4.00,28.00,35.00,42.00,95.00
is_premium_user,"15,000.00",0.30,0.46,0.00,0.00,0.00,1.00,1.00
total_visits,"15,000.00",15.00,3.89,3.00,12.00,15.00,18.00,31.00
avg_session_time,"15,000.00",8.02,2.99,0.01,5.97,7.99,10.06,19.12
pages_per_session,"15,000.00",4.00,1.48,0.01,2.99,4.00,5.01,10.84
email_open_rate,"15,000.00",0.50,0.29,0.00,0.24,0.50,0.75,1.00
email_click_rate,"15,000.00",0.25,0.14,0.00,0.13,0.25,0.38,0.50
total_spent,"13,950.00",524.36,467.05,0.27,300.43,498.84,702.40,"15,910.43"
avg_order_value,"15,000.00",60.08,24.75,0.07,43.03,60.11,76.89,154.55


## 6. Revisión del identificador de clientes

La columna `customer_id` identifica a cada cliente. Se comprueba su cantidad de
valores únicos y si existen identificadores repetidos.

In [8]:
print("Cantidad de registros:", len(df))
print("Clientes únicos:", df["customer_id"].nunique())
print(
    "Identificadores duplicados:",
    df["customer_id"].duplicated().sum()
)

Cantidad de registros: 15000
Clientes únicos: 15000
Identificadores duplicados: 0


## 7. Distribución de la variable objetivo

La variable `churn` indica si un cliente abandonó o no la empresa:

- `0`: no abandonó;
- `1`: abandonó.

Se revisa la cantidad y el porcentaje de clientes pertenecientes a cada clase.

In [9]:
distribucion_churn = df["churn"].value_counts().sort_index()

porcentaje_churn = (
    df["churn"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

resumen_churn = pd.DataFrame({
    "Cantidad": distribucion_churn,
    "Porcentaje": porcentaje_churn
})

display(resumen_churn)

,Cantidad,Porcentaje
churn,,
0,12702,84.68
1,2298,15.32


## 8. Baseline de accuracy

Como existe una clase mayoritaria, se calcula el resultado que obtendría una
regla que predijera siempre la clase más frecuente.

Este valor se utilizará posteriormente como referencia para evaluar los modelos
de clasificación.

In [10]:
clase_mayoritaria = df["churn"].mode()[0]

baseline_accuracy = (
    df["churn"] == clase_mayoritaria
).mean()

print("Clase mayoritaria:", clase_mayoritaria)
print(f"Baseline de accuracy: {baseline_accuracy:.2%}")

Clase mayoritaria: 0
Baseline de accuracy: 84.68%


### Interpretación inicial

El dataset contiene 15.000 registros y cada fila representa un cliente.

La clase `0`, correspondiente a clientes que no abandonaron la empresa, es
considerablemente más frecuente que la clase `1`.

Por este motivo, una accuracy cercana al 84,68 % no necesariamente representa
un buen modelo, ya que ese resultado puede alcanzarse prediciendo siempre la
clase mayoritaria.

Los modelos deberán superar o aportar información adicional respecto de este
baseline y sus resultados serán complementados con una matriz de confusión.